# 03 Model Training

Ziel dieses Notebooks ist es, auf Basis des bereinigten CICIDS2017-Datensatzes ein Machine-Learning-Modell zur binären Klassifikation von Netzwerkverkehr zu trainieren. Dabei wird zwischen normalem Netzwerkverkehr und Angriffen unterschieden.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

In [2]:
PROCESSED_DATA_PATH = Path("../data/processed")
MODEL_OUTPUT_PATH = Path("../outputs/models")

MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

data_file = PROCESSED_DATA_PATH / "cicids2017_friday_binary_clean.csv"

In [3]:
df = pd.read_csv(data_file)

print("Datensatz geladen:", df.shape)
print(df["target"].value_counts())

Datensatz geladen: (702718, 78)
target
0    413933
1    288785
Name: count, dtype: int64


Der bereinigte Datensatz aus der Data-Preparation-Phase wird geladen. Die Zielvariable `target` unterscheidet zwischen normalem Netzwerkverkehr und Angriffen.

## Features und Zielvariable trennen

Für das Modelltraining wird der bereinigte Datensatz in Eingabemerkmale und Zielvariable aufgeteilt.

Die Feature-Matrix `X` enthält alle Netzwerkmerkmale, die dem Modell als Eingabe dienen.  
Die Zielvariable `y` enthält die binäre Klassifikation:

- `0` = BENIGN / normaler Netzwerkverkehr
- `1` = Angriff

Die Spalte `target` wird aus den Eingabemerkmalen entfernt, da sie die vorherzusagende Zielvariable darstellt.

In [4]:
X = df.drop(columns=["target"])
y = df["target"]

print("Feature-Matrix:", X.shape)
print("Zielvariable:", y.shape)

Feature-Matrix: (702718, 77)
Zielvariable: (702718,)


## Aufteilung in Trainings- und Testdaten

Der Datensatz wird in Trainings- und Testdaten aufgeteilt.  
Die Trainingsdaten werden verwendet, um das Modell zu trainieren. Die Testdaten dienen anschließend zur unabhängigen Bewertung der Modellleistung.

Es wird ein Testanteil von 20 % verwendet. Durch `stratify=y` bleibt das Verhältnis zwischen normalem Netzwerkverkehr und Angriffen in Trainings- und Testdaten möglichst gleich.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Trainingsdaten:", X_train.shape)
print("Testdaten:", X_test.shape)

print("\nTrainingsverteilung:")
print(y_train.value_counts())

print("\nTestverteilung:")
print(y_test.value_counts())

Trainingsdaten: (562174, 77)
Testdaten: (140544, 77)

Trainingsverteilung:
target
0    331146
1    231028
Name: count, dtype: int64

Testverteilung:
target
0    82787
1    57757
Name: count, dtype: int64


## Training des Random-Forest-Modells

Für die binäre Klassifikation wird ein Random-Forest-Modell eingesetzt.  
Random Forest eignet sich gut für tabellarische Netzwerkdaten, ist robust gegenüber unterschiedlichen Merkmalsverteilungen und ermöglicht zusätzlich eine erste Interpretation über Feature Importance.

Das Modell soll lernen, normalen Netzwerkverkehr von Angriffen zu unterscheiden.

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Modelltraining abgeschlossen.")

Modelltraining abgeschlossen.


Das Random-Forest-Modell wurde erfolgreich mit den Trainingsdaten trainiert. Es kann nun verwendet werden, um unbekannte Netzwerk-Flows aus den Testdaten als normal oder bösartig zu klassifizieren.

## Erste Vorhersage und Modellcheck

Nach dem Training wird das Modell auf die Testdaten angewendet.  
Die erste Auswertung dient dazu, zu prüfen, ob das Modell grundsätzlich in der Lage ist, normalen Netzwerkverkehr und Angriffe korrekt zu unterscheiden.

Dafür werden die Accuracy und ein Classification Report mit Precision, Recall und F1-Score ausgegeben.

In [7]:
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9988900273224044

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     82787
           1       1.00      1.00      1.00     57757

    accuracy                           1.00    140544
   macro avg       1.00      1.00      1.00    140544
weighted avg       1.00      1.00      1.00    140544



## Speichern des trainierten Modells

Das trainierte Random-Forest-Modell wird gespeichert, damit es in späteren Notebooks erneut geladen und weiterverwendet werden kann.

Dadurch kann das Modell beispielsweise in der Evaluation und in der Explainable-AI-Analyse genutzt werden, ohne es erneut trainieren zu müssen.

In [8]:
model_file = MODEL_OUTPUT_PATH / "random_forest_cicids2017_binary.joblib"

joblib.dump(rf_model, model_file)

print(f"Modell gespeichert unter: {model_file}")

Modell gespeichert unter: ..\outputs\models\random_forest_cicids2017_binary.joblib


## Zusammenfassung Model Training

In diesem Notebook wurde der bereinigte CICIDS2017-Datensatz geladen und in Features sowie Zielvariable aufgeteilt. Anschließend wurden Trainings- und Testdaten erstellt, wobei die Klassenverteilung durch eine stratifizierte Aufteilung erhalten blieb.

Zur Erkennung von Cyberangriffen wurde ein Random-Forest-Modell trainiert. Das Modell unterscheidet zwischen normalem Netzwerkverkehr und Angriffen. Eine erste Vorhersage auf den Testdaten zeigt, ob das Modell grundsätzlich funktioniert. Abschließend wurde das trainierte Modell gespeichert, damit es in der Evaluation und Explainable-AI-Analyse weiterverwendet werden kann.